In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_log_error
from sklearn.ensemble import GradientBoostingRegressor

In [2]:
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

In [3]:
df = pd.read_csv("Superstore.csv")

In [4]:
df.head().T

,0,1,2,3,4
Row ID,1,2,3,4,5
Order ID,CA-2013-152156,CA-2013-152156,CA-2013-138688,US-2012-108966,US-2012-108966
Order Date,09-11-13,09-11-13,13-06-2013,11-10-12,11-10-12
Ship Date,12-11-13,12-11-13,17-06-2013,18-10-2012,18-10-2012
Ship Mode,Second Class,Second Class,Second Class,Standard Class,Standard Class
Customer ID,CG-12520,CG-12520,DV-13045,SO-20335,SO-20335
Customer Name,Claire Gute,Claire Gute,Darrin Van Huff,Sean O'Donnell,Sean O'Donnell
Segment,Consumer,Consumer,Corporate,Consumer,Consumer
Country,United States,United States,United States,United States,United States
City,Henderson,Henderson,Los Angeles,Fort Lauderdale,Fort Lauderdale


In [5]:
df.isna().sum()

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

In [6]:
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

In [7]:
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=True)

In [8]:
df['ship_date'] = pd.to_datetime(df['ship_date'], dayfirst=True)

In [9]:
df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['day'] = df['order_date'].dt.day
df['dayofweek'] = df['order_date'].dt.dayofweek
df['quarter'] = df['order_date'].dt.quarter

In [10]:
df['shipping_days'] = (df['ship_date'] - df['order_date']).dt.days

In [11]:
df.head().T

,0,1,2,3,4
row_id,1,2,3,4,5
order_id,CA-2013-152156,CA-2013-152156,CA-2013-138688,US-2012-108966,US-2012-108966
order_date,2013-11-09 00:00:00,2013-11-09 00:00:00,2013-06-13 00:00:00,2012-10-11 00:00:00,2012-10-11 00:00:00
ship_date,2013-11-12 00:00:00,2013-11-12 00:00:00,2013-06-17 00:00:00,2012-10-18 00:00:00,2012-10-18 00:00:00
ship_mode,Second Class,Second Class,Second Class,Standard Class,Standard Class
customer_id,CG-12520,CG-12520,DV-13045,SO-20335,SO-20335
customer_name,Claire Gute,Claire Gute,Darrin Van Huff,Sean O'Donnell,Sean O'Donnell
segment,Consumer,Consumer,Corporate,Consumer,Consumer
country,United States,United States,United States,United States,United States
city,Henderson,Henderson,Los Angeles,Fort Lauderdale,Fort Lauderdale


In [12]:
drop_cols =[
    'row_id', 'order_id', 'customer_id',
    'customer_name', 'product_id',
    'ship_date', 'ship_mode', 
    'postal_code', 'region'
]

df = df.drop(columns=drop_cols, errors='ignore')

In [13]:
monthly_sales = df.groupby('order_date')['sales'].sum()

In [14]:
monthly_sales = (
    df.groupby([
        pd.Grouper(key='order_date', freq='M'),
        'state',
        'category',
        'sub_category'
    ])
    .agg({
        'sales': 'sum',
        'quantity': 'sum',
        'profit': 'sum',
        'discount': 'mean' 
    })
    .reset_index()
)

In [15]:
order_counts = (
    df.groupby([
        pd.Grouper(key='order_date', freq='M'),
        'state',
        'sub_category'
    ])
    .size()
    .reset_index(name='order_count')
)

In [16]:
monthly_sales = monthly_sales.merge(
    order_counts,
    on=['order_date', 'state', 'sub_category'],
    how='left'
)

In [17]:
monthly_sales['price'] = monthly_sales['sales'] / (monthly_sales['quantity'] + 1e-5)

In [18]:
monthly_sales['profit_margin'] = monthly_sales['profit'] / (monthly_sales['sales'] + 1e-5)

In [19]:
def is_peak_season(date):
    month = date.month
    
    if month in [11, 12]:
        return 1
    
    if month in [6, 7]:
        return 1
    
    return 0

monthly_sales['is_peak_season'] = monthly_sales['order_date'].apply(is_peak_season)

In [20]:
def holiday_weight(date):
    month = date.month
    
    if month == 12: 
        return 3
    elif month == 11:  
        return 2
    elif month == 1:  
        return 1
    else:
        return 0

monthly_sales['holiday_impact'] = monthly_sales['order_date'].apply(holiday_weight)

In [21]:
monthly_sales.head()

,order_date,state,category,sub_category,sales,quantity,profit,discount,order_count,price,profit_margin,is_peak_season,holiday_impact
0,2011-01-31,Arizona,Furniture,Bookcases,181.470,5,-320.5970,0.7,1,36.293927,-1.766667,0,1
1,2011-01-31,Arizona,Office Supplies,Binders,32.340,10,-23.7160,0.7,1,3.233997,-0.733333,0,1
2,2011-01-31,Arizona,Office Supplies,Envelopes,108.720,5,36.6930,0.2,1,21.743957,0.337500,0,1
3,2011-01-31,Arizona,Office Supplies,Paper,56.064,4,19.6224,0.2,1,14.015965,0.350000,0,1
4,2011-01-31,Arkansas,Furniture,Chairs,1067.940,3,224.2674,0.0,1,355.978813,0.210000,0,1


In [22]:
monthly_sales['days_to_christmas'] = (
    pd.to_datetime(monthly_sales['order_date'].dt.year.astype(str) + '-12-25')
    - monthly_sales['order_date']
).dt.days

In [23]:
monthly_sales.head()

,order_date,state,category,sub_category,sales,quantity,profit,discount,order_count,price,profit_margin,is_peak_season,holiday_impact,days_to_christmas
0,2011-01-31,Arizona,Furniture,Bookcases,181.470,5,-320.5970,0.7,1,36.293927,-1.766667,0,1,328
1,2011-01-31,Arizona,Office Supplies,Binders,32.340,10,-23.7160,0.7,1,3.233997,-0.733333,0,1,328
2,2011-01-31,Arizona,Office Supplies,Envelopes,108.720,5,36.6930,0.2,1,21.743957,0.337500,0,1,328
3,2011-01-31,Arizona,Office Supplies,Paper,56.064,4,19.6224,0.2,1,14.015965,0.350000,0,1,328
4,2011-01-31,Arkansas,Furniture,Chairs,1067.940,3,224.2674,0.0,1,355.978813,0.210000,0,1,328


In [24]:
monthly_sales['order_lag_1'] = (
    monthly_sales.groupby(['state', 'sub_category'])['order_count'].shift(1)
)

monthly_sales['order_rolling_3'] = (
    monthly_sales.groupby(['state', 'sub_category'])['order_count']
    .transform(lambda x: x.rolling(3).mean())
)

In [25]:
monthly_sales = monthly_sales.sort_values(['state', 'sub_category', 'order_date',])

monthly_sales['lag_1'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['sales']
    .shift(1)
)

monthly_sales['lag_2'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['sales']
    .shift(2)
)

monthly_sales['lag_12'] = (
    monthly_sales
    .groupby(['state','sub_category'])['sales']
    .shift(12)
)
monthly_sales['rolling_3'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['sales']
    .transform(lambda x: x.rolling(3).mean())
)

In [26]:
monthly_sales['rolling_3'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['sales']
    .transform(lambda x: x.shift(1).rolling(3).mean())
)

monthly_sales['rolling_6'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['sales']
    .transform(lambda x: x.shift(1).rolling(6).mean())
)

monthly_sales['rolling_12'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['sales']
    .transform(lambda x: x.shift(1).rolling(12).mean())
)

In [27]:
monthly_sales['quantity_lag_1'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['quantity']
    .shift(1)
)

monthly_sales['quantity_lag_2'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['quantity']
    .shift(2)
)

monthly_sales['quantity_rolling_3'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['quantity']
    .transform(
        lambda x: x.shift(1).rolling(3).mean()
    )
)

In [28]:
monthly_sales['discount_x_sales'] = (
    monthly_sales['discount'] *
    monthly_sales['sales']
)

In [29]:
monthly_sales['quantity_x_price'] = (
    monthly_sales['quantity'] *
    monthly_sales['price']
)

In [30]:
monthly_sales['profit_lag_1'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['profit']
    .shift(1)
)

In [31]:
monthly_sales['profit_lag_2'] = (
    monthly_sales
    .groupby(['state','category','sub_category'])['profit']
    .shift(2)
)

monthly_sales['profit_lag_12'] = (
    monthly_sales
    .groupby(['state','category','sub_category'])['profit']
    .shift(12)
)

In [32]:
monthly_sales['profit_momentum_1'] = (
    monthly_sales['profit_lag_1'] /
    (monthly_sales['profit_lag_2'] + 1e-5)
)

In [33]:
monthly_sales['profit_rolling_6'] = (
    monthly_sales
    .groupby(['state','category','sub_category'])['profit']
    .transform(
        lambda x: x.shift(1).rolling(6).mean()
    )
)

In [34]:
monthly_sales['discount_lag_1'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['discount']
    .shift(1)
)

In [35]:
monthly_sales['black_friday_flag'] = 0

monthly_sales.loc[
    monthly_sales['order_date'].dt.month == 11,
    'black_friday_flag'
] = 1

In [36]:
monthly_sales['profit_margin_lag_1'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['profit_margin']
    .shift(1)
)

monthly_sales['profit_margin_lag_2'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['profit_margin']
    .shift(2)
)

In [37]:
monthly_sales['order_count_lag_1'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['order_count']
    .shift(1)
)

In [38]:
monthly_sales['order_count_lag_2'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['order_count']
    .shift(2)
)

In [39]:
monthly_sales['order_count_rolling_3'] = (
    monthly_sales
    .groupby(['state', 'sub_category'])['order_count']
    .transform(
        lambda x: x.shift(1).rolling(3).mean()
    )
)

In [40]:
monthly_sales['discount_rolling_3'] = (
    monthly_sales
    .groupby(['state','sub_category'])['discount']
    .transform(lambda x: x.shift(1).rolling(3).mean())
)

In [41]:
monthly_sales['profit_rolling_3'] = (
    monthly_sales
    .groupby(['state','sub_category'])['profit']
    .transform(lambda x: x.shift(1).rolling(3).mean())
)

In [42]:
monthly_sales['quarter'] = (
    monthly_sales['order_date'].dt.quarter
)

In [43]:
monthly_sales['year'] = (
    monthly_sales['order_date'].dt.year
)

In [44]:
monthly_sales['time_index'] = np.arange(len(monthly_sales))

In [45]:
monthly_sales['month'] = monthly_sales['order_date'].dt.month

In [46]:
monthly_sales['month_sin'] = np.sin(2 * np.pi * monthly_sales['month']/12)
monthly_sales['month_cos'] = np.cos(2 * np.pi * monthly_sales['month']/12)

In [47]:
monthly_sales['momentum_1'] = (
    monthly_sales['lag_1'] - monthly_sales['lag_2']
)

In [48]:
monthly_sales = monthly_sales.dropna()

In [49]:
monthly_sales.head()

,order_date,state,category,sub_category,sales,quantity,profit,discount,order_count,price,...,order_count_rolling_3,discount_rolling_3,profit_rolling_3,quarter,year,time_index,month,month_sin,month_cos,momentum_1
4099,2014-02-28,Arizona,Technology,Accessories,31.992,1,4.7988,0.2,1,31.991680,...,1.333333,0.2,9.173667,1,2014,69,2,8.660254e-01,5.000000e-01,-49.936
4415,2014-05-31,Arizona,Technology,Accessories,271.968,4,54.3936,0.2,1,67.991830,...,1.333333,0.2,10.106933,2,2014,70,5,5.000000e-01,-8.660254e-01,-63.992
4577,2014-06-30,Arizona,Technology,Accessories,89.544,7,12.3123,0.2,1,12.791982,...,1.000000,0.2,23.730133,2,2014,71,6,1.224647e-16,-1.000000e+00,239.976
4980,2014-09-30,Arizona,Technology,Accessories,89.584,2,4.4792,0.2,1,44.791776,...,1.000000,0.2,23.834900,3,2014,72,9,-1.000000e+00,-1.836970e-16,-182.424
5352,2014-11-30,Arizona,Technology,Accessories,343.904,16,-33.9044,0.2,3,21.493987,...,1.000000,0.2,23.728367,4,2014,73,11,-5.000000e-01,8.660254e-01,0.040


In [50]:
monthly_sales.columns

Index(['order_date', 'state', 'category', 'sub_category', 'sales', 'quantity',
       'profit', 'discount', 'order_count', 'price', 'profit_margin',
       'is_peak_season', 'holiday_impact', 'days_to_christmas', 'order_lag_1',
       'order_rolling_3', 'lag_1', 'lag_2', 'lag_12', 'rolling_3', 'rolling_6',
       'rolling_12', 'quantity_lag_1', 'quantity_lag_2', 'quantity_rolling_3',
       'discount_x_sales', 'quantity_x_price', 'profit_lag_1', 'profit_lag_2',
       'profit_lag_12', 'profit_momentum_1', 'profit_rolling_6',
       'discount_lag_1', 'black_friday_flag', 'profit_margin_lag_1',
       'profit_margin_lag_2', 'order_count_lag_1', 'order_count_lag_2',
       'order_count_rolling_3', 'discount_rolling_3', 'profit_rolling_3',
       'quarter', 'year', 'time_index', 'month', 'month_sin', 'month_cos',
       'momentum_1'],
      dtype='object')

In [51]:
df['order_date'].min(), df['order_date'].max()

(Timestamp('2011-01-04 00:00:00'), Timestamp('2014-12-31 00:00:00'))

In [52]:
train = monthly_sales[monthly_sales['order_date'] < '2014-12-31']
test = monthly_sales[monthly_sales['order_date'] >= '2014-12-31']

In [53]:
'order_count',  'lag_1', 'rolling_3', 'momentum_1',  'quantity_x_price',

('order_count', 'lag_1', 'rolling_3', 'momentum_1', 'quantity_x_price')

In [54]:
target = 'profit'

categorical_cols = [
    'category',
    'sub_category',
    'state'
]
numerical_cols = [
    'sales',
    'discount',
    'price',

    'profit_momentum_1',
    'discount_x_sales',
    'lag_1', 
    'rolling_3', 
    'quantity_x_price',

    'lag_2',
    'lag_12',

    'rolling_6',
    'rolling_12',

    'discount_rolling_3',
    'profit_rolling_3',

    'quantity_lag_1',
    'quantity_lag_2',
    'quantity_rolling_3',

    'profit_lag_1',

    'discount_lag_1',

    'time_index',
    'month',
    'quarter',
    'year',

    'month_sin',
    'month_cos',

    'is_peak_season',
    'holiday_impact',
    'days_to_christmas',
    'black_friday_flag'
]

X_train = train[categorical_cols + numerical_cols]
y_train = train[target]

X_test = test[categorical_cols + numerical_cols]
y_test = test[target]

In [55]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [56]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numerical_cols)
    ]
)

In [57]:
!pip install catboost

In [58]:
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=1500,
    depth=5,
    learning_rate=0.04,
    l2_leaf_reg=7,
    random_state=42,
    verbose=200
) 

In [59]:
pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ]
)

In [65]:
pipeline.fit(X_train, y_train)

0:	learn: 255.7572395	total: 4.42ms	remaining: 6.62s
200:	learn: 86.7283285	total: 867ms	remaining: 5.61s
400:	learn: 53.7357970	total: 1.73s	remaining: 4.75s
600:	learn: 39.7937225	total: 2.61s	remaining: 3.9s
800:	learn: 30.7605778	total: 3.48s	remaining: 3.04s
1000:	learn: 25.7161104	total: 4.34s	remaining: 2.17s
1200:	learn: 21.7789462	total: 5.12s	remaining: 1.27s
1400:	learn: 19.1498454	total: 6s	remaining: 424ms
1499:	learn: 17.9391951	total: 6.44s	remaining: 0us


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [66]:
from sklearn.metrics import mean_absolute_error, r2_score

preds = pipeline.predict(X_test)

print("MAE:", mean_absolute_error(y_test, preds))
print("R2:", r2_score(y_test, preds))

MAE: 47.88221269187786
R2: 0.817672254620481


In [67]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

In [69]:
mae, rmse, r2

(47.88221269187786, np.float64(154.73345835552365), 0.817672254620481)

In [70]:
joblib.dump(pipeline, "profit_model.pkl")

['profit_model.pkl']

print("Min y_test:", y_test.min())
print("Min prediction:", preds.min())

param_grid = {
    'model__depth': [4, 6, 8],
    'model__learning_rate': [0.03, 0.05, 0.1],
    'model__iterations': [500, 1000, 1500],
    'model__l2_leaf_reg': [3, 5, 7]
}

from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

grid.fit(X_train, y_train)

from sklearn.metrics import mean_absolute_error, r2_score

preds = grid.predict(X_test)

print("MAE:", mean_absolute_error(y_test, preds))
print("R2:", r2_score(y_test, preds))

from sklearn.metrics import mean_squared_error

rmse = np.sqrt(
    mean_squared_error(y_test, preds)
)

print("RMSE:", rmse)


from sklearn.metrics import mean_squared_log_error
rmsle = np.sqrt(mean_squared_log_error(y_test, preds))
print("RMSLE:", rmsle)

In [ ]:
residuals = y_test - preds

error_std = residuals.std()

print("Forecast Error Std:", error_std)

In [ ]:
lower_bound = preds - 1.96 * error_std
upper_bound = preds + 1.96 * error_std

In [ ]:
feature_importance = pipeline.named_steps['model'].get_feature_importance()

feature_names = (
    pipeline.named_steps['preprocessor']
    .get_feature_names_out()
)

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print(importance_df.head(20))